In [1]:
import os
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj, to_dense_batch, to_networkx
from pyvis.network import Network
import networkx as nx

# For reproducibility
torch.manual_seed(42)

# --- 1. GNN Block for Embedding and Pooling ---
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, out_channels)
        self.bn3 = BatchNorm1d(out_channels)

    def forward(self, x, edge_index, batch=None):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        return x

# --- 2. The Main DiffPool Model with SRI-Pool Regularization ---
class DiffPool(torch.nn.Module):
    def __init__(self, dataset, num_clusters):
        super().__init__()
        self.gnn1_embed = GNN(dataset.num_features, 64, 64)
        self.gnn1_pool = GNN(dataset.num_features, 64, num_clusters)
        self.gnn2_embed = GNN(64, 64, 64)
        self.lin1 = Linear(64, 32)
        self.lin2 = Linear(32, dataset.num_classes)

    def forward(self, x, edge_index, batch, return_assignment=False):
        x_embed_1 = self.gnn1_embed(x, edge_index, batch)
        logits = self.gnn1_pool(x, edge_index, batch)
        s_matrix_sparse = F.softmax(logits, dim=-1)
        logit_l1_loss = torch.norm(logits, p=1, dim=-1).mean()
        x_dense, x_mask = to_dense_batch(x_embed_1, batch)
        adj_dense = to_dense_adj(edge_index, batch)
        s_dense, s_mask = to_dense_batch(s_matrix_sparse, batch)
        x_pooled_1 = torch.matmul(s_dense.transpose(-1, -2), x_dense)
        adj_pooled_1 = torch.matmul(torch.matmul(s_dense.transpose(-1, -2), adj_dense), s_dense)
        link_loss = F.mse_loss(torch.matmul(s_dense, s_dense.transpose(-1, -2)), adj_dense)
        eps = 1e-15
        entropy_loss = (-s_matrix_sparse * torch.log(s_matrix_sparse + eps)).sum(dim=-1).mean()
        edge_index_2, edge_weight_2 = self._dense_to_sparse(adj_pooled_1)
        batch_2 = torch.arange(x_pooled_1.size(0), device=x.device).repeat_interleave(x_pooled_1.size(1))
        x_embed_2 = self.gnn2_embed(x_pooled_1.reshape(-1, x_pooled_1.size(-1)), edge_index_2, batch=batch_2)
        readout = global_mean_pool(x_embed_2, batch_2)
        out = F.relu(self.lin1(readout))
        out = self.lin2(out)

        if return_assignment:
            return F.log_softmax(out, dim=-1), s_matrix_sparse
        else:
            return F.log_softmax(out, dim=-1), link_loss, entropy_loss, logit_l1_loss

    def _dense_to_sparse(self, adj):
        edge_indices, edge_weights = [], []
        for i in range(adj.size(0)):
            edge_index = adj[i].nonzero().t()
            edge_weight = adj[i][edge_index[0], edge_index[1]]
            edge_index += i * adj.size(1)
            edge_indices.append(edge_index)
            edge_weights.append(edge_weight)
        return torch.cat(edge_indices, dim=1), torch.cat(edge_weights, dim=0)

# --- 3. Training and Evaluation Setup ---
def train(model, loader, optimizer, lambda_l1, lambda_entropy):
    model.train()
    total_loss, total_link_loss, total_entropy_loss, total_l1_loss = 0, 0, 0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        log_probs, link_loss, entropy_loss, logit_l1_loss = model(data.x, data.edge_index, data.batch)
        class_loss = F.nll_loss(log_probs, data.y)
        loss = class_loss + link_loss + lambda_entropy * entropy_loss + lambda_l1 * logit_l1_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        total_link_loss += link_loss.item() * data.num_graphs
        total_entropy_loss += entropy_loss.item() * data.num_graphs
        total_l1_loss += logit_l1_loss.item() * data.num_graphs
    return (total_loss / len(loader.dataset), total_link_loss / len(loader.dataset),
            total_entropy_loss / len(loader.dataset), total_l1_loss / len(loader.dataset))

@torch.no_grad()
def test(model, loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        log_probs, _, _, _ = model(data.x, data.edge_index, data.batch)
        pred = log_probs.max(dim=1)[1]
        correct += pred.eq(data.y).sum().item()
    return correct / len(loader.dataset)

# --- 4. Visualization Module ---
def visualize_diffpool_clusters(model, data, output_filename="diffpool_visualization.html", threshold=0.1):
    print(f"\nGenerating visualization for a sample graph, saving to {output_filename}...")

    model.eval()
    with torch.no_grad():
        # --- FIX: Manually create the 'batch' tensor for a single graph ---
        # A single Data object has no 'batch' attribute, so we create it.
        # It's a tensor of zeros with length equal to the number of nodes.
        num_nodes = data.num_nodes
        batch = torch.zeros(num_nodes, dtype=torch.long, device=device)

        # Now call the model with the created batch tensor
        _, s_matrix = model(data.x.to(device), data.edge_index.to(device), batch, return_assignment=True)

    nx_graph = to_networkx(data, to_undirected=True)
    net = Network(height="900px", width="100%", bgcolor="#222222", font_color="white", cdn_resources='remote')
    node_color_map = {'original': '#f77f00', 'cluster': '#003049'}
    edge_color_map = {'original': '#8d99ae', 'assignment': '#2a9d8f'}

    for node_idx in range(data.num_nodes):
        net.add_node(f"N_{node_idx}", label=f"N_{node_idx}", title=f"Original Node {node_idx}",
                     color=node_color_map['original'], size=15, group='original_nodes')
    num_clusters = s_matrix.shape[1]
    for cluster_idx in range(num_clusters):
        net.add_node(f"C_{cluster_idx}", label=f"Cluster {cluster_idx}", title=f"Learned Cluster {cluster_idx}",
                     color=node_color_map['cluster'], size=25, group='cluster_nodes')
    for u, v in nx_graph.edges():
        net.add_edge(f"N_{u}", f"N_{v}", color=edge_color_map['original'], width=0.7, dashes=True,
                     title="Original Edge", group='original_edges')
    for node_idx in range(s_matrix.shape[0]):
        for cluster_idx in range(s_matrix.shape[1]):
            weight = s_matrix[node_idx, cluster_idx].item()
            if weight > threshold:
                net.add_edge(f"N_{node_idx}", f"C_{cluster_idx}",
                             title=f"Assignment: {weight:.3f}",
                             width=0.5 + weight * 6,
                             opacity=0.4 + weight * 0.6,
                             color=edge_color_map['assignment'],
                             group='assignment_edges')

    net.toggle_physics(True)
    net.show_buttons(filter_=True)
    net.save_graph(output_filename)
    print(f"Visualization saved successfully. Open '{output_filename}' in your browser to view.")

# --- 5. Main Execution ---
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    dataset_path = os.path.join("..", 'data', 'ENZYMES')
    dataset = TUDataset(dataset_path, name='ENZYMES')
    dataset = dataset.shuffle()
    n = len(dataset) // 10
    test_dataset = dataset[:n]
    train_dataset = dataset[n:]

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    model = DiffPool(dataset=dataset, num_clusters=20).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    LAMBDA_L1 = 0.1
    LAMBDA_ENTROPY = 0.1

    print("Starting training with SRI-Pool regularization...")
    for epoch in range(1, 201):
        train_loss, link_loss, ent_loss, l1_loss = train(model, train_loader, optimizer, LAMBDA_L1, LAMBDA_ENTROPY)
        if epoch % 10 == 0:
            test_acc = test(model, test_loader)
            print(f'Epoch: {epoch:03d}, Total Loss: {train_loss:.4f}, '
                  f'L1 Loss: {l1_loss:.4f}, Test Acc: {test_acc:.4f}')

    print("Training finished.")

    sample_data = test_dataset[0]
    visualize_diffpool_clusters(model, sample_data, output_filename="diffpool_visualization.html", threshold=0.001)

Using device: cuda
Starting training with SRI-Pool regularization...
Epoch: 010, Total Loss: 2.2177, L1 Loss: 4.7970, Test Acc: 0.3333
Epoch: 020, Total Loss: 1.9584, L1 Loss: 3.7003, Test Acc: 0.4500
Epoch: 030, Total Loss: 1.6592, L1 Loss: 2.4000, Test Acc: 0.5167
Epoch: 040, Total Loss: 1.4944, L1 Loss: 2.2017, Test Acc: 0.5500
Epoch: 050, Total Loss: 1.3485, L1 Loss: 1.5541, Test Acc: 0.5500
Epoch: 060, Total Loss: 1.2411, L1 Loss: 1.1966, Test Acc: 0.6167
Epoch: 070, Total Loss: 1.0584, L1 Loss: 0.6345, Test Acc: 0.5667
Epoch: 080, Total Loss: 0.9861, L1 Loss: 0.3223, Test Acc: 0.5833
Epoch: 090, Total Loss: 0.8550, L1 Loss: 0.0789, Test Acc: 0.6833
Epoch: 100, Total Loss: 0.7948, L1 Loss: 0.0269, Test Acc: 0.6167
Epoch: 110, Total Loss: 0.7514, L1 Loss: 0.0102, Test Acc: 0.6667
Epoch: 120, Total Loss: 0.7364, L1 Loss: 0.0027, Test Acc: 0.5667
Epoch: 130, Total Loss: 0.6684, L1 Loss: 0.0031, Test Acc: 0.5333
Epoch: 140, Total Loss: 0.6020, L1 Loss: 0.0027, Test Acc: 0.6667
Epoch: 